# Loan Default Risk Model: Home Credit Data

Predicts which loan applicants are likely to have repayment problems. Uses three tables from the [Home Credit Default Risk](https://www.kaggle.com/c/home-credit-default-risk) Kaggle competition: the main application table, the credit bureau table, and the previous applications table.

**Files needed in a `data/` folder:** `application_train.csv`, `bureau.csv`, `previous_application.csv`.

**Contents**
1. Load and inspect
2. Clean the data
3. Baseline model
4. Add credit bureau and previous application features
5. Add ratio features and compare LightGBM models
6. Remove gender and pick the final model
7. Choose an approval threshold from a cost assumption
8. Explain the model with SHAP

## 1. Load and inspect

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("data/application_train.csv")
print(df.shape)

In [ ]:
# Default rate: the share of applicants with TARGET == 1
df["TARGET"].value_counts(normalize=True)

About 8% of applicants default. A model that approves everyone would be about 92% accurate and catch no defaulters, so accuracy is not a useful metric here. This project uses ROC AUC and AUPRC instead. A model with no skill scores 0.5 on ROC AUC, and its AUPRC equals the default rate.

In [ ]:
# Columns with the most missing values
df.isna().mean().sort_values(ascending=False).head(5)

## 2. Clean the data

`DAYS_EMPLOYED` counts days before the application, so values should be zero or negative. Check it:

In [ ]:
print(df["DAYS_EMPLOYED"].describe())
print()
print(df["DAYS_EMPLOYED"].value_counts().head(3))

The value 365243 (about 1,000 years) is a placeholder for applicants with no employment date, not real data. I replace it with a missing value and keep a flag column so the model can still learn that this group is different.

In [ ]:
df["DAYS_EMPLOYED_ANOMALY"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# Default rate by flag: the two groups behave differently, so the flag is worth keeping
df.groupby("DAYS_EMPLOYED_ANOMALY")["TARGET"].mean()

In [ ]:
# DAYS_BIRTH is a negative number of days; convert to age in years
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365
df["AGE_YEARS"].describe()

## 3. Baseline model

A logistic regression on the main table only, using an identical train/validation split for every model in this notebook (80/20, stratified, `random_state=42`). Numeric columns are filled with the median and scaled. Text columns are filled with the most common value and one-hot encoded. All of this is learned from the training data only, inside a pipeline.

In [ ]:
DROP_COLS = ["TARGET", "SK_ID_CURR", "DAYS_BIRTH"]   # the answer, an ID, and a column replaced by AGE_YEARS

X = df.drop(columns=DROP_COLS)
y = df["TARGET"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("train:", X_train.shape, "| valid:", X_valid.shape)
print("default rate  train:", round(y_train.mean(), 4), "| valid:", round(y_valid.mean(), 4))

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]
print(len(num_cols), "numeric columns,", len(cat_cols), "text columns")

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

baseline = Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))])
baseline.fit(X_train, y_train)

p_base = baseline.predict_proba(X_valid)[:, 1]

In [ ]:
results = []

def record(name, y_true, proba):
    results.append({
        "Model": name,
        "ROC AUC": round(roc_auc_score(y_true, proba), 4),
        "AUPRC": round(average_precision_score(y_true, proba), 4),
    })

record("Logistic regression, main table only", y_valid, p_base)
print("No-skill AUPRC (the default rate):", round(y_valid.mean(), 4))
pd.DataFrame(results)

## 4. Add credit bureau and previous application features

Each of these tables has many rows per applicant. Joining them directly would duplicate applicants, so each table is first summarized to one row per applicant with `groupby`, then joined to the main table with a left join. Row counts are checked after each merge.

In [ ]:
bureau = pd.read_csv("data/bureau.csv")
print(bureau.shape, "| unique applicants:", bureau["SK_ID_CURR"].nunique())

bureau["IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau["IS_OVERDUE"] = (bureau["CREDIT_DAY_OVERDUE"] > 0).astype(int)

bureau_feat = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_N_LOANS=("SK_ID_BUREAU", "count"),
    BUREAU_N_ACTIVE=("IS_ACTIVE", "sum"),
    BUREAU_SHARE_OVERDUE=("IS_OVERDUE", "mean"),
    BUREAU_TOTAL_DEBT=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_MEAN_CREDIT=("AMT_CREDIT_SUM", "mean"),
).reset_index()

print(bureau_feat.shape, "| one row per applicant:", bureau_feat["SK_ID_CURR"].is_unique)

df_bureau = df.merge(bureau_feat, on="SK_ID_CURR", how="left")
print("rows before:", len(df), "| rows after:", len(df_bureau))

In [ ]:
prev = pd.read_csv("data/previous_application.csv")
print(prev.shape, "| unique applicants:", prev["SK_ID_CURR"].nunique())
print(prev["NAME_CONTRACT_STATUS"].value_counts())

prev["IS_REFUSED"] = (prev["NAME_CONTRACT_STATUS"] == "Refused").astype(int)
prev["IS_APPROVED"] = (prev["NAME_CONTRACT_STATUS"] == "Approved").astype(int)

prev_feat = prev.groupby("SK_ID_CURR").agg(
    PREV_N_APPS=("SK_ID_PREV", "count"),
    PREV_SHARE_REFUSED=("IS_REFUSED", "mean"),
    PREV_N_APPROVED=("IS_APPROVED", "sum"),
    PREV_MEAN_CREDIT=("AMT_CREDIT", "mean"),
).reset_index()

df_all = df_bureau.merge(prev_feat, on="SK_ID_CURR", how="left")
print(prev_feat.shape, "| rows before:", len(df_bureau), "| rows after:", len(df_all))

In [ ]:
# Do defaulters look different on the new features?
df_all.groupby("TARGET")[["BUREAU_N_ACTIVE", "BUREAU_SHARE_OVERDUE",
                          "PREV_N_APPS", "PREV_SHARE_REFUSED", "PREV_N_APPROVED"]].mean()

## 5. Ratio features and LightGBM

LightGBM builds many small decision trees, each correcting the previous ones. It handles missing values and text categories directly and captures interactions that a linear model cannot. The helper function below trains it on any version of the data, using the same split as the baseline.

The ratio features are common lending measures:

| Feature | Meaning |
|---|---|
| `CREDIT_INCOME_RATIO` | loan amount relative to income |
| `ANNUITY_INCOME_RATIO` | yearly payment relative to income (payment burden) |
| `CREDIT_TERM` | yearly payment relative to loan amount (roughly the inverse of the term) |
| `GOODS_CREDIT_RATIO` | goods price relative to loan amount (like loan-to-value) |
| `EMPLOYED_TO_AGE` | share of life spent in the current job |
| `EXT_SOURCE_MEAN` | average of the three external credit scores |

In [ ]:
def fit_lgbm(data):
    X = data.drop(columns=DROP_COLS)
    y = data["TARGET"]
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    num = X_tr.select_dtypes(include="number").columns
    cats = [c for c in X_tr.columns if c not in num]
    for c in cats:
        X_tr[c] = X_tr[c].astype("category")
        X_va[c] = pd.Categorical(X_va[c], categories=X_tr[c].cat.categories)
    model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                               random_state=42, n_jobs=-1, verbose=-1)
    model.fit(X_tr, y_tr)
    return model, X_tr, X_va, y_va, model.predict_proba(X_va)[:, 1]

_, _, _, y_v1, p_lgb_bureau = fit_lgbm(df_bureau)
record("LightGBM + credit bureau features", y_v1, p_lgb_bureau)

_, _, _, y_v2, p_lgb_prev = fit_lgbm(df_all)
record("+ previous application features", y_v2, p_lgb_prev)
pd.DataFrame(results)

In [ ]:
df_fe = df_all.copy()
df_fe["CREDIT_INCOME_RATIO"] = df_fe["AMT_CREDIT"] / df_fe["AMT_INCOME_TOTAL"]
df_fe["ANNUITY_INCOME_RATIO"] = df_fe["AMT_ANNUITY"] / df_fe["AMT_INCOME_TOTAL"]
df_fe["CREDIT_TERM"] = df_fe["AMT_ANNUITY"] / df_fe["AMT_CREDIT"]
df_fe["GOODS_CREDIT_RATIO"] = df_fe["AMT_GOODS_PRICE"] / df_fe["AMT_CREDIT"]
df_fe["EMPLOYED_TO_AGE"] = df_fe["DAYS_EMPLOYED"] / df_fe["DAYS_BIRTH"]
df_fe["EXT_SOURCE_MEAN"] = df_fe[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)

model_fe, X_train_fe, X_valid_fe, y_valid_fe, p_fe = fit_lgbm(df_fe)
record("+ ratio features (payment burden, loan vs. goods price)", y_valid_fe, p_fe)
pd.DataFrame(results)

## 6. Remove gender and pick the final model

In an earlier SHAP run on the model with all features, `CODE_GENDER` ranked among the most influential features. In the US, sex is a protected attribute in credit decisions, so a lender would not use it as an input. This section checks what removing it costs.

In [ ]:
df_final = df_fe.drop(columns=["CODE_GENDER"])
model_final, X_train_final, X_valid_final, y_valid_final, p_final = fit_lgbm(df_final)
record("Final model (gender removed)", y_valid_final, p_final)

results_table = pd.DataFrame(results)
results_table

Removing gender changes the scores very little, so the model did not need it. Dropping a column does not remove other features that may correlate with it, and no group-level fairness audit was done here.

## 7. Choose an approval threshold from a cost assumption

The model outputs a default probability. Approving or denying requires a cutoff. The best cutoff depends on what each mistake costs:

- A **missed default** (approving someone who defaults) costs the unpaid loan.
- A **wrongly denied good customer** costs the interest that would have been earned.

I assume a missed default costs 5 times as much as a wrongly denied customer. **This 5:1 ratio is an assumption for illustration.** A real lender would supply its own figures.

In [ ]:
def cost_sweep(y_true, proba, cost_fn=5, cost_fp=1):
    rows = []
    for thr in np.arange(0.02, 0.60, 0.01):
        flagged = proba >= thr
        fn = int(((~flagged) & (y_true == 1)).sum())
        fp = int((flagged & (y_true == 0)).sum())
        rows.append((round(thr, 2), flagged.mean(), fn * cost_fn + fp * cost_fp))
    sweep = pd.DataFrame(rows, columns=["threshold", "share_denied", "cost"])
    approve_all = int((y_true == 1).sum()) * cost_fn
    best = sweep.loc[sweep["cost"].idxmin()]
    return sweep, best, approve_all

for name, y_true, proba in [("Logistic baseline", y_valid, p_base),
                            ("Final model", y_valid_final, p_final)]:
    sweep, best, approve_all = cost_sweep(y_true, proba)
    print(f"{name}: best threshold {best['threshold']:.2f} | "
          f"share denied {best['share_denied']:.1%} | "
          f"cost reduction vs. approving everyone {1 - best['cost'] / approve_all:.1%}")

In [ ]:
sweep_final, best_final, _ = cost_sweep(y_valid_final, p_final)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep_final["threshold"], sweep_final["cost"], color="#4C78A8")
ax.axvline(best_final["threshold"], ls="--", color="gray")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Total cost (assumed units)")
ax.set_title("Cost vs. threshold, final model")
plt.show()

## 8. Explain the model with SHAP

SHAP splits each prediction into the contribution of every feature. The summary plot shows what drives risk overall (red is a high feature value, blue is low; right means higher predicted risk). The waterfall plot explains one applicant: it starts at the average prediction and shows each feature pushing the risk up (red) or down (blue).

In [ ]:
import shap

sample = X_valid_final.sample(2000, random_state=42)
explainer = shap.TreeExplainer(model_final)
sv = explainer(sample)

shap.summary_plot(sv, sample, max_display=10)

In [ ]:
proba = model_final.predict_proba(sample)[:, 1]
i = int(np.argmax(proba))      # the riskiest applicant in the sample
print("Predicted default probability:", round(float(proba[i]), 3))

shap.plots.waterfall(sv[i], max_display=10)

## Limitations

- Only 3 of the 8 source tables are used.
- Model variants were compared on the same validation split, so the final scores are slightly optimistic. Cross-validation or a separate test set would be more rigorous.
- The 5:1 cost ratio is an assumption, not a business figure.
- Removing gender does not remove features that may correlate with it, and no group-level fairness audit was done.
- Scores come from my own validation split and are not Kaggle leaderboard scores.